# Chronos-Bolt Patch-Sweep — Waveform Inspector

**Role of this notebook.** This is the *interactive, per-frequency* companion to
`frequency_sweep.py`. The script is the **quantitative results-of-record** (it sweeps a
whole frequency grid, averages over phases, and writes `metrics.csv` + the recovery/phase
curves). This notebook is for the thing the script can't do: **look at the actual predicted
waveform vs ground truth for one case**, so you can *see why* a feature in the sweep exists.

- Change **`FREQ`** in the config cell and re-run the *Inspect one frequency* cell.
- The **official** model (`amazon/chronos-bolt-tiny`) serves as the **p16-s16 data point**
  (200k steps, full corpus) — it is the strongest S=16 anchor for the stride comparison.
- Metrics reported are **amplitude recovery** (pred/GT sinusoid amplitude, 1.0 = perfect)
  and **phase error**, not a single MSE — a lone MSE at one frequency/phase is misleading.
- The probe signal is built with the project's own `tones.make_tone` (same convention the
  TSMixup/KernelSynth generators inject), and each model's **`cpp = freq·P/fs`** is shown —
  patch-aliasing nulls fall at **integer cpp**.

In [ ]:
# %pip install chronos-forecasting   # uncomment on a fresh environment

import sys
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, torch
from chronos import ChronosBoltPipeline

# reuse the project's canonical tone + cpp helpers (single source of truth with the generators)
for _base in [Path.cwd(), *Path.cwd().parents]:
    _cand = _base / "chronos" / "data" / "synthetic"
    if not _cand.exists():
        _cand = _base / "data" / "synthetic"
    if _cand.exists():
        sys.path.insert(0, str(_cand)); break
from tones import make_tone, cpp as cpp_of

# local training outputs — load weights from here (instant) instead of downloading from HF
LOCAL_MODELS = _cand.parent.parent / "outputs" / "models"   # chronos/outputs/models

# ---------- what to inspect (edit FREQ, then re-run the inspector cell) ----------
FREQ              = 22.0     # signal frequency [Hz] to inspect
PHASE             = 0.0      # signal phase [rad]

# ---------- fixed probe geometry ----------
FS                = 512      # sampling frequency [Hz]
AMPLITUDE         = 5.0      # sinusoid amplitude (normalized away by the model; used for GT sanity)
CONTEXT_LENGTH    = 512      # history fed to the model (trained native = 2048; 512 gives ~same result)
PREDICTION_LENGTH = 64       # forecast horizon [samples] (<= trained 64)
QUANTILE          = 4        # index into [0.1..0.9]; 4 = 0.5 median (the point forecast)

# ---------- models ----------
REPO           = "federicosabbadini/chronos-bolt-patch-sweep"
OFFICIAL_MODEL = "amazon/chronos-bolt-tiny"
OFFICIAL_LABEL = "p16-s16 (official)"   # official IS the p16-s16 data point

MODEL_NAMES    = [
    "p16-s12-seed42", "p16-s8-seed42",
    "p16-s4-seed42", "p8-s8-seed42", "p24-s24-seed42",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("results"); OUTPUT_DIR.mkdir(exist_ok=True)
print("device:", DEVICE)

In [ ]:
# ---------- helpers ----------
def ps(name):
    """'p16-s8-seed42' -> (16, 8); 'p16-s16 (official)' -> (16, 16)."""
    import re
    m = re.search(r"p(\d+)-s(\d+)", name)
    return (int(m.group(1)), int(m.group(2))) if m else (16, 16)

def generate_sinusoid(freq, phase=0.0, n=None):
    """Clean sinusoid over context+horizon samples; returns (t_seconds, y).
    Reuses tones.make_tone so the probe matches the synthetic generators' convention."""
    n = n or (CONTEXT_LENGTH + PREDICTION_LENGTH)
    t = np.arange(n) / FS
    return t, make_tone(freq, FS, n, amplitude=AMPLITUDE, phase=phase)

def fit_sinusoid(y, t, freq):
    """LS-fit y ~ a*cos + b*sin + c at `freq`. Returns (amplitude, phase).
    Robust for short / non-integer-period horizons where an FFT bin is too coarse."""
    X = np.stack([np.cos(2*np.pi*freq*t), np.sin(2*np.pi*freq*t), np.ones_like(t)], axis=1)
    (a, b, _c), *_ = np.linalg.lstsq(X, y, rcond=None)
    return float(np.hypot(a, b)), float(np.arctan2(b, a))

def wrap(angle):
    """Wrap a phase error to [-pi, pi]."""
    return float((angle + np.pi) % (2*np.pi) - np.pi)

@torch.no_grad()
def median_forecast(pipe, context_1d):
    ctx = torch.tensor(context_1d, dtype=torch.float32, device=DEVICE)
    return pipe.predict(ctx, prediction_length=PREDICTION_LENGTH)[:, QUANTILE, :].float().cpu().numpy().flatten()

In [ ]:
# ---------- load every model ONCE (re-run only if you change the model list) ----------
def load_variant(name):
    # local training output if present (instant), else HF
    local = LOCAL_MODELS / name
    if (local / "model.safetensors").exists():
        return ChronosBoltPipeline.from_pretrained(str(local), device_map=DEVICE, torch_dtype=torch.float32)
    return ChronosBoltPipeline.from_pretrained(REPO, subfolder=name, device_map=DEVICE, torch_dtype=torch.float32)

print("loading models ...")
MODELS = {OFFICIAL_LABEL: ChronosBoltPipeline.from_pretrained(OFFICIAL_MODEL, device_map=DEVICE)}
for name in MODEL_NAMES:
    MODELS[name] = load_variant(name)
print("loaded:", list(MODELS))

## Inspect one frequency

Edit `FREQ` in the config cell, re-run this cell. Prints amplitude recovery / phase error /
RMSE per model, then plots the waveforms (all models colored equally — the official serves
as the P=16 S=16 data point, not a separate reference).

In [ ]:
t, sig = generate_sinusoid(FREQ, PHASE)
context, gt = sig[:CONTEXT_LENGTH], sig[CONTEXT_LENGTH:]
t_fut     = np.arange(CONTEXT_LENGTH, CONTEXT_LENGTH + PREDICTION_LENGTH)
t_fut_sec = t[CONTEXT_LENGTH:]
amp_gt, ph_gt = fit_sinusoid(gt, t_fut_sec, FREQ)

# ---- metrics table ----
print(f"FREQ={FREQ} Hz | period={FS/FREQ:.1f} samples | horizon={PREDICTION_LENGTH} samples "
      f"| patch nulls k*fs/P: P16@{FS/16:.0f},{2*FS/16:.0f}  P8@{FS/8:.0f}  P24@{FS/24:.0f},{2*FS/24:.0f} Hz\n")
print(f"{'model':24s} {'cpp':>6s} {'recovery':>9s} {'phase_err':>10s} {'RMSE':>8s}")
preds = {}
for name, pipe in MODELS.items():
    p = median_forecast(pipe, context); preds[name] = p
    amp, ph = fit_sinusoid(p, t_fut_sec, FREQ)
    rec = amp / amp_gt if amp_gt > 1e-8 else np.nan
    pe  = np.degrees(abs(wrap(ph - ph_gt)))
    cval = cpp_of(FREQ, ps(name)[0], FS)
    print(f"{name:24s} {cval:6.2f} {rec:9.3f} {pe:9.1f}\u00b0 {np.sqrt(np.mean((p-gt)**2)):8.3f}")

# ---- waveform plot: context tail + horizon ----
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
tail = 128
ax[0].plot(np.arange(CONTEXT_LENGTH-tail, CONTEXT_LENGTH), context[-tail:], color="black", lw=1.5, label="context")
ax[0].plot(t_fut, gt, color="green", lw=2.5, label="ground truth")
for name in MODELS:
    ax[0].plot(t_fut, preds[name], "--", lw=1.8, alpha=0.8, label=name)
ax[0].axvline(CONTEXT_LENGTH, color="grey", ls=":", alpha=0.6)
ax[0].set_title(f"Context tail + forecast @ {FREQ} Hz"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)

# ---- horizon-only zoom ----
ax[1].plot(gt, color="green", lw=2.5, label="ground truth")
for name in MODELS:
    ax[1].plot(preds[name], "--", lw=1.8, alpha=0.8, label=name)
ax[1].set_title(f"Forecast horizon zoom @ {FREQ} Hz"); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / f"inspect_{FREQ:.0f}Hz.png", dpi=150, bbox_inches="tight"); plt.show()

## Uncertainty band for one model

The median alone hides how (un)confident a model is. This shows the q0.1–q0.9 interval for a
single chosen variant at the current `FREQ`.

In [ ]:
INSPECT_MODEL = OFFICIAL_LABEL   # <- change to any name in MODELS

with torch.no_grad():
    ctx = torch.tensor(context, dtype=torch.float32, device=DEVICE)
    q = MODELS[INSPECT_MODEL].predict(ctx, prediction_length=PREDICTION_LENGTH)[0].float().cpu().numpy()  # [Q, H]

plt.figure(figsize=(12, 4))
plt.fill_between(np.arange(PREDICTION_LENGTH), q[0], q[8], alpha=0.25, label="q0.1\u2013q0.9")
plt.plot(q[QUANTILE], lw=2, label="median")
plt.plot(gt, color="green", lw=2, label="ground truth")
plt.title(f"{INSPECT_MODEL} @ {FREQ} Hz \u2014 median + uncertainty"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

## Multi-frequency panel

Inspect several diagnostic frequencies at once — e.g. an easy low frequency, the ~22 Hz
resonance, and the P=16 patch nulls at 32 / 64 Hz — to see the waveform behaviour side by side.

In [ ]:
INSPECT_FREQS = [9, 22, 32, 64]

fig, axes = plt.subplots(len(INSPECT_FREQS), 1, figsize=(13, 2.6*len(INSPECT_FREQS)), sharex=True)
for ax, f in zip(axes, INSPECT_FREQS):
    _, s = generate_sinusoid(f, PHASE)
    c, g = s[:CONTEXT_LENGTH], s[CONTEXT_LENGTH:]
    ax.plot(g, color="green", lw=2.5, label="ground truth")
    for name in MODELS:
        ax.plot(median_forecast(MODELS[name], c), "--", lw=1.5, alpha=0.8, label=name)
    ax.set_title(f"{f} Hz  |  period {FS/f:.1f} samples  |  cpp@P16={cpp_of(f, 16, FS):.2f}"); ax.grid(alpha=0.3)
axes[0].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "multi_frequency_panel.png", dpi=150, bbox_inches="tight"); plt.show()

---
### For the quantitative sweep

Run the full frequency response (grid + phase averaging + curves + `metrics.csv`):

```
python chronos/eval/frequency_sweep.py
```

Outputs land in `chronos/eval/results/`. Use this notebook to drill into any
frequency the sweep flags.